# 네 가지 모델을 한 번에 비교하기

`1_1` ~ `1_3` 에서 모델을 하나씩 불러 봤습니다. 이제 **네 개를 한꺼번에** 돌려 비교합니다.

## 이 노트북의 핵심 생각

모델을 **코드가 아니라 데이터로** 다룹니다.

```python
models = {
    "OpenAI": (openai, "gpt-4.1-mini"),
    "Gemini": (gemini, "gemini-2.5-flash"),
    ...
}
```

이렇게 해 두면 모델을 추가할 때 **딕셔너리에 한 줄만 넣으면 됩니다.**
`if` 문을 늘려 가며 분기하지 않아도 됩니다.

## 흐름

| 단계 | 내용 |
|---|---|
| Step 1~5 | 클라이언트 네 개 만들기 |
| Step 6~7 | 질문과 모델 목록 준비 |
| Step 8~9 | 한 번에 돌리기 |
| Step 10~11 | 결과를 파일로 남기기 |

## Step 1. 가져오기

여기서도 `openai` 하나로 네 모델을 모두 부릅니다.
`requests` 는 지금 쓰지 않지만 뒤에서 직접 HTTP 를 다룰 때를 위해 남겨 두었습니다.

In [1]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

## Step 2. 키 세 개 읽기

```python
openai_api_key    = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key    = os.getenv('GOOGLE_API_KEY')
```

Ollama 키가 없는 이유는 **필요 없기 때문**입니다 (`1_3_ollama` Step 4 참고).

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

## Step 3. 키가 다 있는지 한눈에 확인

세 줄 모두 `exists` 로 나와야 뒤가 정상 동작합니다.
키마다 앞부분 모양이 달라 **어느 회사 키인지 구분**할 수 있습니다.

| 회사 | 앞부분 |
|---|---|
| OpenAI | `sk-proj-` |
| Anthropic | `sk-ant-` |
| Google | `AI` 등 |

> 없는 키가 있어도 괜찮습니다. Step 9 에서 그 모델만 오류로 표시되고 나머지는 돌아갑니다.

In [3]:
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ


## Step 4. OpenAI 클라이언트

기본값을 쓰므로 괄호가 비어 있습니다.

In [4]:
openai = OpenAI()

## Step 5-1. 나머지 회사들의 주소

`.env` 에 세 주소를 적어 두었습니다.

```
ANTHROPIC_URL=https://api.anthropic.com/v1/
GEMINI_URL=https://generativelanguage.googleapis.com/v1beta/openai/
OLLAMA_URL=http://localhost:11434/v1
```

In [5]:
anthropic_url = os.getenv('ANTHROPIC_URL')
gemini_url = os.getenv('GEMINI_URL')
ollama_url = os.getenv('OLLAMA_URL')

## Step 5-2. 클라이언트 세 개를 더 만든다

```python
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini    = OpenAI(api_key=google_api_key,    base_url=gemini_url)
ollama    = OpenAI(api_key="ollama",          base_url=ollama_url)
```

**세 줄 모두 `OpenAI(...)` 입니다.** 변수 이름만 다릅니다.

이것이 가능한 이유는 세 회사가 모두 OpenAI 호환 입구를 제공하기 때문입니다.
덕분에 아래 비교 함수를 **회사마다 따로 짜지 않아도** 됩니다.

In [10]:
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url) if google_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

print("anthropic:", "준비됨" if anthropic else "키 없음 (건너뜀)")
print("gemini:", "준비됨" if gemini else "키 없음 (건너뜀)")

anthropic: 키 없음 (건너뜀)
gemini: 준비됨


## Step 6. 질문 하나를 준비

```python
message = [{"role": "user", "content": "저는 AI 보안에 관심이 많습니다. ..."}]
```

**"1개만 적어 주세요"** 라고 못 박았습니다.
비교가 목적이므로 답이 길면 읽기 어렵기 때문입니다.

> 모델을 비교할 때는 **답의 형식을 좁게 지정**하는 편이 좋습니다.
> 그래야 내용 차이가 드러납니다.

In [12]:
message = [
    {"role": "user", 
     "content": "저는 AI 보안에 관심이 많습니다. AI 보안에 대해 가장 심각하게 생각하는 위협 1개만 적어 주세요 "},
]

## Step 7. 모델 목록 — 이 노트북의 핵심

```python
models = {
    "OpenAI": (openai,    "gpt-4.1-mini"),
    "Gemini": (gemini,    "gemini-2.5-flash"),
    "Claude": (anthropic, "claude-haiku-4-5-20251001"),
    "Ollama": (ollama,    "llama3.2"),
}
```

`{표시이름: (클라이언트, 모델명)}` 모양의 딕셔너리입니다.

이 구조 덕분에 모델을 추가할 때 **이 딕셔너리에 한 줄만** 넣으면 됩니다.
아래 함수는 고칠 필요가 없습니다.

> 모델 이름에 날짜가 붙은 것(`claude-haiku-4-5-20251001`)은 **버전 고정**입니다.
> 모델이 조용히 바뀌어 결과가 달라지는 일을 막아 줍니다.

In [13]:
models = {
    "OpenAI": (openai, "gpt-4.1-mini"),
    "Gemini": (gemini, "gemini-3.8-flash"),
    "Claude": (anthropic, "claude-haiku-4-5-20251001"),
    "Ollama": (ollama, "llama3.2"),
}


## Step 8. 비교 함수

```python
for name, (client, model) in models.items():
    ...
    try:
        response = client.chat.completions.create(model=model, messages=message)
    except Exception as e:
        print(f"[Error] {e}")
        responses[name] = None
```

### `try` / `except` 가 중요한 이유

모델 네 개 중 **하나가 실패해도 나머지는 계속 돌아야** 합니다.

- Ollama 서버가 안 떠 있을 수 있고
- 키가 하나 빠져 있을 수 있고
- 한 회사 API 만 일시적으로 장애일 수 있습니다

`except` 가 없으면 첫 실패에서 전체가 멈추고, **이미 받은 답도 잃어버립니다.**
실패한 모델은 `None` 으로 남겨 두고 넘어갑니다.

In [14]:
def response_compare(message):
    responses = {}
    for name, (client, model) in models.items():
        print(f"===== {name} ({model}) =====")
        try:
            response = client.chat.completions.create(model=model, messages=message)
            content = response.choices[0].message.content
            print(content)
            responses[name] = content
        except Exception as e:
            print(f"[Error] {e}")
            responses[name] = None
        print()
    return responses

## Step 9. 한 번에 실행

네 모델이 차례로 답합니다. 출력을 비교하며 읽어 보세요.

| 보는 곳 | 무엇을 볼까 |
|---|---|
| **길이** | 짧게 답하라고 했는데 지킨 모델은? |
| **형식** | 굵은 글씨·목록을 쓰는 정도 |
| **내용** | 같은 위협을 꼽았나, 다른 것을 꼽았나 |
| **한국어** | 자연스러운가 |

> 실제로 OpenAI·Gemini·Claude 는 모두 **적대적 공격(Adversarial Attack)** 을 꼽았습니다.
> 서로 다른 회사 모델이 비슷한 답을 내는 것도 흥미로운 관찰입니다.
> 반면 로컬 모델의 한국어는 눈에 띄게 거칩니다.

In [15]:
results = response_compare(message)

===== OpenAI (gpt-4.1-mini) =====
[Error] Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

===== Gemini (gemini-3.8-flash) =====
AI 보안 분야에서 현재 가장 심각하고 파괴적인 위협 하나를 꼽는다면 단연 **‘데이터 오염 공격(Data Poisoning Attack, 특히 백도어 공격)’**입니다.

### 선정 이유: "인공지능의 뇌 자체를 은밀하게 오염시키기 때문"

이 위협이 가장 치명적인 이유는 다음과 같습니다.

1. **완벽한 위장성 (Detection Difficulty)**
   공격자가 학습 데이터나 파인튜닝 데이터셋에 악의적인 패턴(트리거)을 심어두면, 모델은 평상시에는 벤치마크 테스트에서 99%의 정확도를 보이며 완벽하게 정상 작동합니다. 따라서 배포 전 보안 검사를 쉽게 통과합니다.
2. **치명적인 영구성 (High Remediation Cost)**
   특정 '트리거(단어, 이미지의 특정 픽셀 등)'가 입력되는 순간에만 모델이 공격자가 원하는 오작동(예: 자율주행차의 정지 신호 무시, 악성코드 탐지 우회, 보안 시스템 무력화)을 일으킵니다. 이를 발견하더라도 수십~수백억 개의 파라미터와 방대한 데이터 중에서 무엇이 오염되었는지 찾아내기 불가능에 가까워, **결국 막대한 비용을 들여 처음부터 다시 학습시켜야 합니다.**
3. **AI 공급망(Supply Chain)의 붕괴**
   현대 AI 개발은 허깅페이스(Hugging Face) 등 오픈소스 모델이나 외부 크롤링 

## Step 10. 결과를 파일로 남기는 함수

모델 비교는 **나중에 다시 보려고** 하는 일입니다.
노트북 출력은 다시 실행하면 사라지므로 마크다운 파일로 저장해 둡니다.

```python
lines = ["# LLM Response Comparison", "", f"_Generated: {...}_", ...]
```

실행 시각을 함께 적는 것이 중요합니다.
**모델은 계속 바뀝니다.** 두 달 뒤 같은 코드를 돌리면 다른 답이 나올 수 있어서,
언제 받은 답인지 적어 두지 않으면 비교가 무의미해집니다.

```python
with open(filepath, "w", encoding="utf-8") as f:
```

`encoding="utf-8"` 을 빼면 윈도우에서 **한글이 깨집니다.** 꼭 넣으세요.

In [16]:
from datetime import datetime

def save_comparison_markdown(message, results, filepath="comparison.md"):
    user_prompt = next((m["content"] for m in message if m["role"] == "user"), "")

    lines = [
        "# LLM Response Comparison",
        "",
        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",
        "",
        "## Prompt",
        "",
        user_prompt,
        "",
    ]

    for name, content in results.items():
        model = models[name][1]
        lines.append(f"## {name} ({model})")
        lines.append("")
        lines.append(content if content else "_No response (see error during generation)_")
        lines.append("")

    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"Saved comparison to {filepath}")
    return filepath

## Step 11. 저장 실행

`comparison.md` 파일이 생깁니다. 열어서 네 모델의 답을 나란히 비교해 보세요.

In [17]:
save_comparison_markdown(message, results)

Saved comparison to comparison.md


'comparison.md'

---

## 정리

| | 내용 |
|---|---|
| **모델을 데이터로** | 딕셔너리에 담으면 추가가 한 줄로 끝난다 |
| **실패를 견디게** | `try`/`except` 로 한 모델이 죽어도 나머지는 살린다 |
| **결과를 남긴다** | 모델은 바뀐다. 시각과 함께 저장해 둔다 |
| **`encoding="utf-8"`** | 윈도우에서 한글이 깨지지 않게 |

### 직접 해 보기

1. `models` 에 `gpt-5-nano` 를 한 줄 추가해 보세요. 함수를 안 고쳐도 되는지 확인해 보세요.
2. 질문을 "코드를 짜 주세요" 로 바꿔 보세요. 순위가 달라지나요?
3. Ollama 서버를 끄고 실행해 보세요. 나머지 세 개는 정상인가요?

### 다음 노트북

| 노트북 | 내용 |
|---|---|
| `1_5_tokenizer` | 토큰이 정확히 무엇이고 모델마다 왜 다른가 |
| `1_6_stateless` | 모델이 기억하지 못한다는 것의 의미 |